In [ ]:
# === Setup Environment ===
import sys, os
if 'google.colab' in sys.modules:
    # Run on Colab
    os.system('git clone https://github.com/ptquanh/Olympic-AI-From-Scratch.git')
    sys.path.append('Olympic-AI-From-Scratch')
elif os.path.exists('/kaggle/working'):
    # Run on Kaggle
    os.system('git clone https://github.com/ptquanh/Olympic-AI-From-Scratch.git')
    sys.path.append('Olympic-AI-From-Scratch')

# Runtime: ~1 phút
import random
import numpy as np
random.seed(42)
np.random.seed(42)
print('Setup completed!')

# 1. Logistic Regression (From Scratch)

Khác với Linear Regression trả về một số thực vô hạn, **Logistic Regression** dùng hàm **Sigmoid** để ép kết quả về khoảng $(0, 1)$, từ đó dùng làm xác suất phân loại nhị phân (Binary Classification).

Hàm Loss ở đây là **Binary Cross-Entropy (Log Loss)**.

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from sklearn.datasets import make_classification

# Tạo dữ liệu phân loại nhị phân
X, y = make_classification(n_samples=200, n_features=2, n_informative=2, 
                           n_redundant=0, n_clusters_per_class=1, random_state=42)

plt.scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', alpha=0.7)
plt.xlabel('Feature 1')
plt.ylabel('Feature 2')
plt.title('Dữ liệu phân loại (Đỏ=0, Xanh=1)')
plt.show()

## Bước 1: Khởi tạo mô hình
Hàm Sigmoid: $\sigma(z) = \frac{1}{1 + e^{-z}}$
Mô hình dự đoán xác suất: $\hat{y} = \sigma(XW + b)$

In [ ]:
class LogisticRegressionFromScratch:
    def __init__(self, learning_rate=0.01, n_iterations=1000):
        self.lr = learning_rate
        self.n_iters = n_iterations
        self.W = None
        self.b = None
        self.loss_history = []
        
    def sigmoid(self, z):
        return 1 / (1 + np.exp(-z))
        
    def fit(self, X, y):
        n_samples, n_features = X.shape
        
        # Khởi tạo tham số ngẫu nhiên
        self.W = np.zeros(n_features)
        self.b = 0
        
        # Gradient Descent
        for i in range(self.n_iters):
            # Forward pass: Tính z và xác suất dự đoán (y_pred)
            z = np.dot(X, self.W) + self.b
            y_pred = self.sigmoid(z)
            
            # Tính Loss (Binary Cross-Entropy)
            # Thêm 1e-9 để tránh lỗi log(0)
            loss = -(1/n_samples) * np.sum(y * np.log(y_pred + 1e-9) + (1-y) * np.log(1-y_pred + 1e-9))
            self.loss_history.append(loss)
            
            # Tính Gradients
            # Rất kỳ diệu là đạo hàm của CrossEntropy + Sigmoid giống hệt MSE của Linear Regression!
            # dL/dW = (1/N) * X^T * (y_pred - y)
            dW = (1/n_samples) * np.dot(X.T, (y_pred - y))
            db = (1/n_samples) * np.sum(y_pred - y)
            
            # Cập nhật tham số
            self.W -= self.lr * dW
            self.b -= self.lr * db
            
            if (i+1) % 200 == 0:
                print(f'Iteration {i+1}: Loss = {loss:.4f}')
                
    def predict_proba(self, X):
        z = np.dot(X, self.W) + self.b
        return self.sigmoid(z)
    
    def predict(self, X, threshold=0.5):
        probs = self.predict_proba(X)
        return [1 if p >= threshold else 0 for p in probs]

In [ ]:
# Chạy thử mô hình
model = LogisticRegressionFromScratch(learning_rate=0.1, n_iterations=1000)
model.fit(X, y)


## Trực quan hóa Decision Boundary

In [ ]:
x_min, x_max = X[:, 0].min() - 1, X[:, 0].max() + 1
y_min, y_max = X[:, 1].min() - 1, X[:, 1].max() + 1
xx, yy = np.meshgrid(np.arange(x_min, x_max, 0.02),
                     np.arange(y_min, y_max, 0.02))

Z = np.array(model.predict(np.c_[xx.ravel(), yy.ravel()]))
Z = Z.reshape(xx.shape)

plt.contourf(xx, yy, Z, alpha=0.3, cmap='bwr')
plt.scatter(X[:, 0], X[:, 1], c=y, cmap='bwr', edgecolor='k')
plt.title('Decision Boundary (From Scratch)')
plt.show()